# Operatori di creazione e distruzione di time-step

In [163]:
from QuantumCompiling import CircuitState, CircuitShape
import scipy.sparse as sp
import numpy as np

Definizione della classe CircuitOperations, che calcola e tiene in memoria gli operatori |n><m| per spostarsi dal generico time-step m al generico time-step n 

In [164]:
class CircuitOperations:
    def __init__(self,shape):  
        self._hops = np.zeros((len(shape._steps),len(shape._steps)),dtype=object)
        for i in range(len(shape._steps)):
            for j in range(len(shape._steps)):
                self._hops[i,j] = sp.csr_matrix( np.kron(
                    np.conj(shape.state_of_step(shape._steps[j])).reshape(-1,1),
                    shape.state_of_step(shape._steps[i])
                )  )

    # moves the single-step-circuit state to "to" single-step-circuit state
    def hop(self,to,circuit):
        if not circuit.get_n_steps() == 1: raise Exception("The circuit is not single-step")
        circuit._state = self._hops[circuit._shape.index_of_state(circuit._state),to].dot(circuit._state) 

Si carica la solita shape

In [165]:
n_qubits = 4
gates = { 
    1 : [ ["ID"]],
    2 : [ ["Sc","St"],["CZc","CZt"] ]
    ,3 : [ ["Sc","ID","St"] ]
    ,4 : [ ["Sc","Sc","St","St"] ] 
}

shape = CircuitShape(n_qubits,gates)
shape.print_steps(); 

0: ['ID', 'ID', 'ID', 'ID']
1: ['ID', 'ID', 'Sc', 'St']
2: ['ID', 'ID', 'CZc', 'CZt']
3: ['ID', 'Sc', 'St', 'ID']
4: ['ID', 'CZc', 'CZt', 'ID']
5: ['ID', 'Sc', 'ID', 'St']
6: ['Sc', 'St', 'ID', 'ID']
7: ['Sc', 'St', 'Sc', 'St']
8: ['Sc', 'St', 'CZc', 'CZt']
9: ['CZc', 'CZt', 'ID', 'ID']
10: ['CZc', 'CZt', 'Sc', 'St']
11: ['CZc', 'CZt', 'CZc', 'CZt']
12: ['Sc', 'ID', 'St', 'ID']
13: ['Sc', 'Sc', 'St', 'St']



E si prova, creato un CircuitOperations su tale shape, a spostare lo stato di un generico circuito single-step a ad un generico circuito single-step

In [166]:
c = CircuitOperations(shape)

steps1 = [
    ['CZc', 'CZt', 'ID', 'ID']
]
    
c1 = CircuitState(shape, steps1)

print("Before:",c1.draw())
c.hop(13,c1)
print("After:",c1.draw())

Before:         
q_0: ─■─
      │ 
q_1: ─■─
        
q_2: ───
        
q_3: ───
        
After:            
q_0: ─X────
      │    
q_1: ─┼──X─
      │  │ 
q_2: ─X──┼─
         │ 
q_3: ────X─
           
